# 01 · Validación, limpieza, EDA y entrega de base para modelado

Este notebook prepara la base de datos inicial del proyecto **BiciMAD Predictor**.

El objetivo es validar las fuentes de datos, descargar los datos necesarios, limpiar la información, integrar variables temporales y meteorológicas, realizar un análisis exploratorio y dejar documentada la base que deberá usar la fase posterior de preprocesamiento y modelado.

### Qué hace este notebook

- Identifica las fuentes de datos utilizadas.
- Descarga las fuentes necesarias en `data/raw`.
- Limpia datasets auxiliares.
- Limpia el histórico principal de estaciones BiciMAD 2022.
- Integra calendario laboral de Madrid.
- Integra meteorología histórica de AEMET.
- Crea una base enriquecida intermedia en `data/interim`.
- Genera una copia de entrada para modelado en `data/processed`.
- Define reglas de limpieza para EDA.
- Analiza riesgos de estaciones casi vacías y casi llenas.
- Deja claro qué base debe usar el siguiente notebook.

### Qué no hace este notebook

- No entrena modelos.
- No define el modelo final.
- No calcula métricas predictivas.
- No realiza división train/test.
- No aplica codificación ni escalado final de variables.
- No crea el target definitivo de Machine Learning.
- No genera un dataset final entrenable; solo entrega una base preparada para que el siguiente notebook construya el dataset de modelado.

La salida recomendada para que continúe la persona encargada de modelado es:

`data/processed/station_status_history_2022_modeling_base.csv`

La versión trazable del pipeline queda conservada en:

`data/interim/bicimad/station_status_history_2022_enriched_base.csv`


## Índice maestro del notebook

0. Portada y objetivo del notebook  
1. Resumen rápido para el equipo de modelado  
2. Explicación sencilla del problema y del dataset principal  
3. Mapa general del flujo de datos  
4. Inventario de fuentes originales  
5. Qué papel cumple cada dataset  
6. Configuración del proyecto  
7. Seguridad, credenciales y archivos protegidos  
8. Descarga o verificación de datos originales  
9. Validación inicial de datos raw  
10. Limpieza de datasets auxiliares  
    10.1 GBFS estaciones actuales  
    10.2 GBFS estado actual  
    10.3 Histórico diario de viajes  
    10.4 Calendario laboral  
    10.5 Meteorología AEMET  
11. Limpieza del histórico principal BiciMAD 2022  
12. Creación de la base enriquecida intermedia  
13. Validación de la base enriquecida  
14. Contrato de datos para modelado  
15. Reglas de limpieza para EDA  
16. EDA global  
17. EDA por estación  
18. EDA temporal  
19. EDA meteorológica  
20. EDA estación + hora  
21. Hallazgos principales  
22. Limitaciones  
23. Entrega para el siguiente notebook  
24. Archivos generados


## 1. Resumen rápido para el equipo de modelado

El archivo que debe usar la persona encargada de preprocesamiento y modelado es:

`data/processed/station_status_history_2022_modeling_base.csv`

Este archivo contiene registros históricos de estaciones de BiciMAD durante 2022, enriquecidos con calendario laboral de Madrid y meteorología diaria observada de AEMET.

Cada fila representa el estado de una estación concreta de BiciMAD en un momento concreto del tiempo.

> **Cada fila responde a la pregunta:**  
> ¿Cómo estaba una estación concreta de BiciMAD en una fecha y hora concretas?

La versión equivalente y trazable dentro del pipeline de limpieza se guarda también en:

`data/interim/bicimad/station_status_history_2022_enriched_base.csv`

La diferencia entre ambos archivos es de propósito:

- `data/interim/...`: archivo intermedio trazable, útil para auditar la limpieza y el enriquecimiento.
- `data/processed/...`: copia recomendada como punto de entrada para el siguiente notebook de modelado.

Esta base todavía **no es el dataset final de Machine Learning**. No contiene una variable objetivo definitiva, no tiene división train/test y no tiene codificación ni escalado final de variables.

El siguiente notebook deberá partir de esta base para definir el problema predictivo concreto, crear el target, seleccionar variables, dividir los datos temporalmente y entrenar modelos.


### 1.1 Base recomendada para la fase de preprocesamiento y modelado

| Elemento | Valor |
|---|---|
| Archivo recomendado para modelado | `data/processed/station_status_history_2022_modeling_base.csv` |
| Archivo intermedio trazable | `data/interim/bicimad/station_status_history_2022_enriched_base.csv` |
| Granularidad | Estación + momento temporal |
| Identificador principal | `station_id_historical` |
| Fecha/hora principal | `snapshot_datetime` |
| Variables temporales | `date`, `snapshot_hour`, `snapshot_day_of_week`, `snapshot_month` |
| Variables de calendario | `day_type`, `is_working_day`, `is_weekend`, `is_holiday` |
| Variables meteorológicas | temperatura, precipitación, humedad |
| Variables operativas | `num_bikes_available`, `num_docks_available`, `occupation_ratio` |
| Ubicación para uso del equipo de modelado | `data/processed` |
| Responsabilidad del siguiente notebook | crear target, preprocesar variables, dividir datos y entrenar modelos |

Este notebook no decide el target final ni entrena modelos. Su responsabilidad es dejar una base histórica limpia, validada, enriquecida y bien documentada para que el siguiente paso pueda trabajar con seguridad.


## 2. Explicación sencilla del problema y del dataset principal

BiciMAD es un sistema de bicicleta pública. En este tipo de sistema pueden aparecer dos problemas operativos principales:

1. Una estación puede quedarse con muy pocas bicicletas disponibles.
2. Una estación puede quedarse con muy pocos anclajes libres para devolver bicicletas.

Estos problemas no ocurren igual en todas las estaciones ni en todas las horas. Algunas estaciones pueden quedarse casi vacías en determinadas franjas horarias, mientras que otras pueden saturarse y dificultar la devolución de bicicletas.

Por eso, el proyecto necesita una base histórica que permita estudiar patrones por:

- estación,
- fecha,
- hora,
- tipo de día,
- condiciones meteorológicas,
- disponibilidad de bicicletas,
- disponibilidad de anclajes.

La base principal del proyecto se construye a partir del histórico de estado de estaciones de BiciMAD de 2022. A ese histórico se le añaden variables de calendario y meteorología para que el análisis y el futuro modelo tengan más contexto.


## 3. Mapa general del flujo de datos

El camino de los datos en este notebook sigue esta lógica:

~~~text
Datos originales en data/raw
│
├── Histórico de estaciones BiciMAD 2022
│   └── Limpieza
│       └── station_status_history_2022_clean.csv
│
├── Calendario laboral de Madrid
│   └── Limpieza
│       └── madrid_working_calendar_clean.csv
│
├── Clima diario AEMET 2022
│   └── Limpieza y agregación
│       └── aemet_daily_weather_madrid_2022_join_ready.csv
│
└── Unión por fecha
    └── data/interim/bicimad/station_status_history_2022_enriched_base.csv
        └── copia para modelado
            └── data/processed/station_status_history_2022_modeling_base.csv
~~~

No todos los datasets descargados se usan para entrenar un modelo. Algunos sirven para enriquecer el histórico, otros para contexto de negocio y otros para una posible demo futura.

El archivo recomendado para que continúe el equipo de modelado es:

`data/processed/station_status_history_2022_modeling_base.csv`


## 4. Inventario de fuentes originales

Antes de limpiar o transformar datos, es importante saber de dónde viene cada fuente y para qué sirve.

Esta sección crea un inventario de datasets del proyecto. La tabla distingue entre:

- fuentes principales para análisis y modelado,
- fuentes auxiliares para enriquecer datos,
- fuentes útiles para contexto, visualización o demo futura.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from urllib.parse import quote
from io import BytesIO
import json
import os
import re
import sys
import time
import zipfile
import platform
import warnings

import pandas as pd
import numpy as np

try:
    import requests
except ImportError as exc:
    raise ImportError(
        "Falta la librería requests. Instálala con: pip install requests"
    ) from exc

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib no está disponible. El notebook seguirá sin gráficos.")

from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)


@dataclass
class DataSource:
    """Representa una fuente de datos utilizada en el proyecto."""

    name: str
    source_type: str
    origin: str
    granularity: str
    project_use: str
    raw_file: str
    interim_file: str
    modeling_role: str


class DataSourceRegistry:
    """Inventario de fuentes de datos del proyecto."""

    def __init__(self, sources: list[DataSource]):
        self.sources = sources

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(
            [
                {
                    "Dataset": source.name,
                    "Tipo": source.source_type,
                    "Origen": source.origin,
                    "Granularidad": source.granularity,
                    "Uso en el proyecto": source.project_use,
                    "Archivo raw": source.raw_file,
                    "Archivo intermedio": source.interim_file,
                    "Papel para modelado": source.modeling_role,
                }
                for source in self.sources
            ]
        )


data_sources = [
    DataSource(
        name="Histórico de estaciones BiciMAD 2022",
        source_type="Fuente principal",
        origin="EMT Madrid / BiciMAD",
        granularity="Estación + momento temporal",
        project_use="Base principal para analizar disponibilidad y saturación por estación.",
        raw_file="data/raw/bicimad/bicimad_station_status_history_2022_raw.zip",
        interim_file="data/interim/bicimad/station_status_history_2022_clean.csv",
        modeling_role="Fuente base para construir el dataset de modelado.",
    ),
    DataSource(
        name="Calendario laboral de Madrid",
        source_type="Fuente auxiliar",
        origin="Portal de datos abiertos del Ayuntamiento de Madrid",
        granularity="Día",
        project_use="Añadir contexto de día laborable, sábado, domingo o festivo.",
        raw_file="data/raw/calendar/madrid_working_calendar_raw.csv",
        interim_file="data/interim/calendar/madrid_working_calendar_clean.csv",
        modeling_role="Enriquece la base principal con variables temporales.",
    ),
    DataSource(
        name="Climatología diaria AEMET 2022",
        source_type="Fuente auxiliar",
        origin="AEMET OpenData",
        granularity="Día + estación meteorológica",
        project_use="Añadir temperatura, precipitación y humedad observada al histórico 2022.",
        raw_file="data/raw/weather/aemet_daily_climate_selected_stations_2022_raw.json",
        interim_file="data/interim/weather/aemet_daily_weather_madrid_2022_join_ready.csv",
        modeling_role="Enriquece la base principal con variables meteorológicas históricas.",
    ),
    DataSource(
        name="GBFS estaciones actuales BiciMAD",
        source_type="Fuente de referencia actual",
        origin="GBFS BiciMAD",
        granularity="Estación actual",
        project_use="Referencia actual de estaciones, coordenadas, capacidad y atributos.",
        raw_file="data/raw/bicimad/gbfs_station_information_raw.json",
        interim_file="data/interim/bicimad/stations_clean.csv",
        modeling_role="No es la base de entrenamiento histórica; útil para demo, mapa o referencia actual.",
    ),
    DataSource(
        name="GBFS estado actual BiciMAD",
        source_type="Fuente de referencia actual",
        origin="GBFS BiciMAD",
        granularity="Foto actual de estación",
        project_use="Consultar disponibilidad actual de bicicletas y anclajes.",
        raw_file="data/raw/bicimad/gbfs_station_status_snapshot_raw.json",
        interim_file="data/interim/bicimad/station_status_snapshot_clean.csv",
        modeling_role="No se usa para entrenar el histórico 2022; útil para demo o comparación futura.",
    ),
    DataSource(
        name="Histórico diario de viajes BiciMAD",
        source_type="Fuente de contexto",
        origin="EMT Madrid / BiciMAD",
        granularity="Día",
        project_use="Entender la evolución general del uso de BiciMAD.",
        raw_file="data/raw/bicimad/bicimad_daily_trips_raw.csv",
        interim_file="data/interim/bicimad/bicimad_daily_trips_clean.csv",
        modeling_role="Contexto de negocio; no sustituye al histórico por estación.",
    ),
    DataSource(
        name="Inventario de estaciones AEMET",
        source_type="Fuente auxiliar",
        origin="AEMET OpenData",
        granularity="Estación meteorológica",
        project_use="Seleccionar estaciones meteorológicas cercanas y fiables para Madrid.",
        raw_file="data/raw/weather/aemet_stations_inventory_raw.json",
        interim_file="data/interim/weather/aemet_stations_inventory_clean.csv",
        modeling_role="Apoya la preparación de variables meteorológicas históricas.",
    ),
    DataSource(
        name="Predicción horaria AEMET Madrid",
        source_type="Fuente para demo futura",
        origin="AEMET OpenData",
        granularity="Hora futura",
        project_use="Demostrar cómo una app futura podría consumir meteorología prevista.",
        raw_file="data/raw/weather/aemet_forecast_hourly_madrid_raw.json",
        interim_file="data/interim/weather/aemet_forecast_hourly_madrid_clean.csv",
        modeling_role="No se usa para entrenar el histórico 2022; sirve para demo o integración futura.",
    ),
]

registry = DataSourceRegistry(data_sources)
data_sources_df = registry.to_dataframe()
display(data_sources_df)


## 5. Qué papel cumple cada dataset

El proyecto utiliza varios datasets, pero no todos tienen el mismo papel.

La fuente central es el histórico de estado de estaciones de BiciMAD 2022. Esta fuente indica cuántas bicicletas y anclajes había disponibles en cada estación a lo largo del tiempo.

El calendario laboral y la meteorología diaria no sustituyen a BiciMAD, sino que lo enriquecen. Sirven para añadir contexto: no es lo mismo analizar una estación en un día laborable que en un festivo, ni en un día frío que en un día templado.

Las fuentes GBFS actuales representan una fotografía reciente de la red. Son útiles para una demo, un mapa o una aplicación futura, pero no deben mezclarse automáticamente con el histórico de 2022 sin un trabajo adicional de compatibilidad.

La predicción horaria de AEMET es útil para una posible app futura, porque permite consultar el tiempo previsto. Sin embargo, no se usa para entrenar el histórico de 2022, ya que el entrenamiento necesita variables históricas observadas.

En resumen:

- **Dataset recomendado para el siguiente paso:** `data/processed/station_status_history_2022_modeling_base.csv`.
- **Dataset intermedio trazable:** `data/interim/bicimad/station_status_history_2022_enriched_base.csv`.
- **Datasets auxiliares:** calendario laboral y clima diario observado.
- **Datasets de contexto o demo:** GBFS actual, viajes diarios y predicción horaria.


## 6. Configuración del proyecto

Antes de descargar o transformar datos, necesitamos comprobar que el notebook se está ejecutando dentro del repositorio correcto y que las carpetas principales existen.


In [ ]:
@dataclass
class ProjectConfig:
    """Configuración central de rutas del proyecto."""

    project_root: Path

    @property
    def data_dir(self) -> Path:
        return self.project_root / "data"

    @property
    def raw_dir(self) -> Path:
        return self.data_dir / "raw"

    @property
    def interim_dir(self) -> Path:
        return self.data_dir / "interim"

    @property
    def processed_dir(self) -> Path:
        return self.data_dir / "processed"

    @property
    def notebooks_dir(self) -> Path:
        return self.project_root / "notebooks"

    @property
    def src_dir(self) -> Path:
        return self.project_root / "src"

    @property
    def models_dir(self) -> Path:
        return self.project_root / "models"

    @property
    def reports_dir(self) -> Path:
        return self.project_root / "reports"

    @property
    def figures_dir(self) -> Path:
        return self.reports_dir / "figures"

    @property
    def raw_bicimad_dir(self) -> Path:
        return self.raw_dir / "bicimad"

    @property
    def raw_calendar_dir(self) -> Path:
        return self.raw_dir / "calendar"

    @property
    def raw_weather_dir(self) -> Path:
        return self.raw_dir / "weather"

    @property
    def interim_bicimad_dir(self) -> Path:
        return self.interim_dir / "bicimad"

    @property
    def interim_calendar_dir(self) -> Path:
        return self.interim_dir / "calendar"

    @property
    def interim_weather_dir(self) -> Path:
        return self.interim_dir / "weather"

    @property
    def expected_directories(self) -> list[Path]:
        return [
            self.data_dir,
            self.raw_dir,
            self.raw_bicimad_dir,
            self.raw_calendar_dir,
            self.raw_weather_dir,
            self.interim_dir,
            self.interim_bicimad_dir,
            self.interim_calendar_dir,
            self.interim_weather_dir,
            self.processed_dir,
            self.notebooks_dir,
            self.src_dir,
            self.models_dir,
            self.reports_dir,
            self.figures_dir,
        ]


class ProjectRootFinder:
    """Localiza la raíz del repositorio a partir de la carpeta actual."""

    @staticmethod
    def find(start_path: Path | None = None) -> Path:
        current_path = start_path or Path.cwd()

        for path in [current_path, *current_path.parents]:
            if (path / ".git").exists():
                return path

        raise FileNotFoundError(
            "No se ha podido localizar la raíz del repositorio. "
            "Asegúrate de ejecutar este notebook dentro del proyecto clonado."
        )


project_root = ProjectRootFinder.find()
config = ProjectConfig(project_root=project_root)

for directory in config.expected_directories:
    directory.mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto detectada correctamente:")
print(config.project_root)

print("\nInformación básica del entorno:")
print(f"Python: {sys.version.split()[0]}")
print(f"Sistema operativo: {platform.platform()}")
print(f"Carpeta actual: {Path.cwd()}")


## 7. Seguridad, credenciales y archivos protegidos

El proyecto usa la API de AEMET para descargar datos meteorológicos.

La clave real de AEMET debe guardarse en un archivo local llamado `.env`.

Ese archivo no debe subirse a GitHub porque contiene información privada. El repositorio incluye `.env.example` como plantilla para que cada persona cree su propio `.env`.

Este notebook nunca imprime la clave real. Solo comprueba si existe y si parece estar configurada.

### 7.1 Cómo obtener y configurar la API Key de AEMET

Página oficial de AEMET OpenData:  
https://opendata.aemet.es/

Página oficial para obtener la API Key:  
https://opendata.aemet.es/centrodedescargas/obtencionAPIKey

### 7.2 Crear el archivo `.env` de forma segura

Ejecuta en terminal desde la raíz del repositorio:

~~~bash
read -r -s -p "Introduce tu AEMET_API_KEY: " AEMET_API_KEY
echo
umask 077
printf "AEMET_API_KEY=%s\n" "$AEMET_API_KEY" > .env
unset AEMET_API_KEY
chmod 600 .env
echo ".env creado con permisos seguros."
ls -l .env
~~~

### 7.3 Confirmar que `.env` no aparece en Git

~~~bash
git status -sb
~~~

Si `.gitignore` está bien configurado, `.env` no debe aparecer como archivo pendiente.


In [ ]:
class EnvironmentManager:
    """Gestiona comprobaciones básicas de archivos de entorno sin mostrar secretos."""

    def __init__(self, config: ProjectConfig):
        self.config = config
        self.env_path = self.config.project_root / ".env"
        self.env_example_path = self.config.project_root / ".env.example"

    def env_file_exists(self) -> bool:
        return self.env_path.exists()

    def env_example_exists(self) -> bool:
        return self.env_example_path.exists()

    def read_env_variable(self, variable_name: str) -> str | None:
        if not self.env_file_exists():
            return None

        for line in self.env_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()

            if not line or line.startswith("#") or "=" not in line:
                continue

            key, value = line.split("=", 1)

            if key.strip() == variable_name:
                return value.strip()

        return None

    def check_aemet_api_key(self) -> dict:
        api_key = self.read_env_variable("AEMET_API_KEY")

        return {
            ".env existe": self.env_file_exists(),
            ".env.example existe": self.env_example_exists(),
            "AEMET_API_KEY configurada": bool(api_key),
            "Longitud de AEMET_API_KEY": len(api_key) if api_key else 0,
        }


class GitIgnoreChecker:
    """Comprueba que .gitignore protege los elementos críticos del proyecto."""

    def __init__(self, config: ProjectConfig):
        self.gitignore_path = config.project_root / ".gitignore"

    def check_patterns(self, patterns: list[str]) -> pd.DataFrame:
        content = self.gitignore_path.read_text(encoding="utf-8") if self.gitignore_path.exists() else ""

        return pd.DataFrame(
            [
                {
                    "Patrón": pattern,
                    "Encontrado en .gitignore": pattern in content,
                }
                for pattern in patterns
            ]
        )


environment_manager = EnvironmentManager(config)
environment_status = environment_manager.check_aemet_api_key()

print("Estado de configuración de AEMET:")
display(pd.DataFrame([environment_status]))

if environment_status["AEMET_API_KEY configurada"]:
    print("La clave de AEMET está configurada localmente. No se muestra por seguridad.")
else:
    print("La clave de AEMET no está configurada todavía en .env. Las descargas de AEMET quedarán pendientes.")

protected_patterns = [
    ".env",
    "data/raw/**",
    "data/interim/**",
    "data/processed/**",
    "models/**",
    ".ipynb_checkpoints/",
]

gitignore_checker = GitIgnoreChecker(config)
display(gitignore_checker.check_patterns(protected_patterns))


## 8. Descarga o verificación de datos originales

En esta sección se descargan o verifican las fuentes originales que se usarán en el proyecto.

Los datos originales se guardan en `data/raw` y no se modifican directamente. Algunas fuentes pueden pesar bastante, especialmente el histórico de BiciMAD 2022. Esto es normal en un proyecto real de datos.

Los datos descargados no se suben a GitHub porque están protegidos por `.gitignore`.

### 8.1 Parámetros de ejecución


In [ ]:
RUN_DOWNLOADS = True
RUN_HEAVY_DOWNLOADS = True
FORCE_REDOWNLOAD = False

HTTP_TIMEOUT_SECONDS = 180
AEMET_WAIT_SECONDS = 1.0

print("Parámetros de ejecución:")
print(f"RUN_DOWNLOADS: {RUN_DOWNLOADS}")
print(f"RUN_HEAVY_DOWNLOADS: {RUN_HEAVY_DOWNLOADS}")
print(f"FORCE_REDOWNLOAD: {FORCE_REDOWNLOAD}")
print("\nAviso: algunas fuentes pueden pesar bastante. Los archivos se guardarán localmente en data/raw.")


### 8.2 Catálogo de fuentes raw esperadas


In [ ]:
@dataclass(frozen=True)
class RawDatasetSpec:
    dataset: str
    source_owner: str
    source_url: str
    local_path: Path
    requires_api_key: bool
    is_heavy_download: bool
    required_for_modeling_base: bool
    access_note: str


class RawDatasetCatalog:
    def __init__(self, specs: list[RawDatasetSpec]):
        self.specs = specs

    def to_dataframe(self) -> pd.DataFrame:
        records = []
        for spec in self.specs:
            exists = spec.local_path.exists()
            size_mb = round(spec.local_path.stat().st_size / 1024 / 1024, 2) if exists else 0
            records.append(
                {
                    "Dataset": spec.dataset,
                    "Fuente": spec.source_owner,
                    "URL o endpoint de referencia": spec.source_url,
                    "Ruta raw esperada": str(spec.local_path.relative_to(config.project_root)),
                    "Existe localmente": exists,
                    "Tamaño MB": size_mb,
                    "Requiere API Key": spec.requires_api_key,
                    "Descarga pesada": spec.is_heavy_download,
                    "Necesario para base de modelado": spec.required_for_modeling_base,
                    "Nota de acceso": spec.access_note,
                }
            )
        return pd.DataFrame(records)


raw_dataset_specs = [
    RawDatasetSpec(
        dataset="GBFS estaciones actuales BiciMAD",
        source_owner="EMT Madrid / BiciMAD",
        source_url="https://madrid.publicbikesystem.net/customer/gbfs/v2/es/station_information",
        local_path=config.raw_bicimad_dir / "gbfs_station_information_raw.json",
        requires_api_key=False,
        is_heavy_download=False,
        required_for_modeling_base=False,
        access_note="Feed GBFS actual. Útil para referencia, mapa o demo.",
    ),
    RawDatasetSpec(
        dataset="GBFS estado actual BiciMAD",
        source_owner="EMT Madrid / BiciMAD",
        source_url="https://madrid.publicbikesystem.net/customer/gbfs/v2/es/station_status",
        local_path=config.raw_bicimad_dir / "gbfs_station_status_snapshot_raw.json",
        requires_api_key=False,
        is_heavy_download=False,
        required_for_modeling_base=False,
        access_note="Feed GBFS actual. Fotografía de disponibilidad.",
    ),
    RawDatasetSpec(
        dataset="Histórico diario de viajes BiciMAD",
        source_owner="EMT Madrid / BiciMAD",
        source_url="https://datos.emtmadrid.es/dataset/a3442c30-ee33-434f-998d-c63a51a8c446/resource/883ba442-76f4-4969-b5b2-1e9d50df2f86/download/historicoviajesdia.csv",
        local_path=config.raw_bicimad_dir / "bicimad_daily_trips_raw.csv",
        requires_api_key=False,
        is_heavy_download=False,
        required_for_modeling_base=False,
        access_note="Serie diaria de viajes. Sirve como contexto de negocio.",
    ),
    RawDatasetSpec(
        dataset="Histórico de estaciones BiciMAD 2022",
        source_owner="EMT Madrid / BiciMAD",
        source_url="https://media.emtmadrid.es/-uHaW6iZkhG",
        local_path=config.raw_bicimad_dir / "bicimad_station_status_history_2022_raw.zip",
        requires_api_key=False,
        is_heavy_download=True,
        required_for_modeling_base=True,
        access_note="Descarga pesada. Se usará la parte de estado de estaciones 2022.",
    ),
    RawDatasetSpec(
        dataset="Calendario laboral de Madrid",
        source_owner="Ayuntamiento de Madrid",
        source_url="https://datos.madrid.es/dataset/300082-0-calendario_laboral/resource/300082-1-calendario_laboral-csv/download/300082-1-calendario_laboral-csv.csv",
        local_path=config.raw_calendar_dir / "madrid_working_calendar_raw.csv",
        requires_api_key=False,
        is_heavy_download=False,
        required_for_modeling_base=True,
        access_note="Calendario laboral 2013-2026.",
    ),
    RawDatasetSpec(
        dataset="Inventario de estaciones AEMET",
        source_owner="AEMET OpenData",
        source_url="https://opendata.aemet.es/opendata/api/valores/climatologicos/inventarioestaciones/todasestaciones",
        local_path=config.raw_weather_dir / "aemet_stations_inventory_raw.json",
        requires_api_key=True,
        is_heavy_download=False,
        required_for_modeling_base=False,
        access_note="Permite seleccionar estaciones meteorológicas cercanas y fiables.",
    ),
    RawDatasetSpec(
        dataset="Climatología diaria AEMET 2022",
        source_owner="AEMET OpenData",
        source_url="https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{fecha_inicio}/fechafin/{fecha_fin}/estacion/{indicativo}",
        local_path=config.raw_weather_dir / "aemet_daily_climate_selected_stations_2022_raw.json",
        requires_api_key=True,
        is_heavy_download=False,
        required_for_modeling_base=True,
        access_note="Datos meteorológicos diarios observados de 2022 para estaciones seleccionadas.",
    ),
    RawDatasetSpec(
        dataset="Predicción horaria AEMET Madrid",
        source_owner="AEMET OpenData",
        source_url="https://opendata.aemet.es/opendata/api/prediccion/especifica/municipio/horaria/28079",
        local_path=config.raw_weather_dir / "aemet_forecast_hourly_madrid_raw.json",
        requires_api_key=True,
        is_heavy_download=False,
        required_for_modeling_base=False,
        access_note="Predicción horaria para demo futura.",
    ),
]

raw_catalog = RawDatasetCatalog(raw_dataset_specs)
display(raw_catalog.to_dataframe())


### 8.3 Funciones de descarga


In [ ]:
class LocalFileDownloader:
    """Descarga archivos públicos a rutas locales."""

    def __init__(self, timeout_seconds: int = 180):
        self.timeout_seconds = timeout_seconds

    def download(self, url: str, output_path: Path, force: bool = False) -> bool:
        output_path.parent.mkdir(parents=True, exist_ok=True)

        if output_path.exists() and not force:
            print(f"Ya existe: {output_path.relative_to(config.project_root)}")
            return False

        print(f"Descargando: {url}")
        print(f"Destino: {output_path.relative_to(config.project_root)}")

        with requests.get(url, stream=True, timeout=self.timeout_seconds) as response:
            response.raise_for_status()
            with output_path.open("wb") as file:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        file.write(chunk)

        size_mb = output_path.stat().st_size / 1024 / 1024
        print(f"Descarga completada: {size_mb:.2f} MB")
        return True


class AEMETOpenDataClient:
    """Cliente mínimo para descargar datos desde AEMET OpenData."""

    def __init__(self, api_key: str, timeout_seconds: int = 180, wait_seconds: float = 1.0):
        if not api_key:
            raise ValueError("No se ha encontrado AEMET_API_KEY en .env.")
        self.api_key = api_key
        self.timeout_seconds = timeout_seconds
        self.wait_seconds = wait_seconds

    def request_metadata(self, endpoint: str) -> dict:
        response = requests.get(
            endpoint,
            params={"api_key": self.api_key},
            timeout=self.timeout_seconds,
        )
        response.raise_for_status()
        return response.json()

    def download_endpoint_json(self, endpoint: str):
        metadata = self.request_metadata(endpoint)
        if "datos" not in metadata:
            raise ValueError(f"La respuesta de AEMET no contiene clave 'datos': {metadata}")
        data_url = metadata["datos"]
        data_response = requests.get(data_url, timeout=self.timeout_seconds)
        data_response.raise_for_status()
        time.sleep(self.wait_seconds)
        return data_response.json()

    def download_to_file(self, endpoint: str, output_path: Path, force: bool = False) -> bool:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        if output_path.exists() and not force:
            print(f"Ya existe: {output_path.relative_to(config.project_root)}")
            return False
        print(f"Descargando AEMET: {endpoint}")
        data = self.download_endpoint_json(endpoint)
        output_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
        size_mb = output_path.stat().st_size / 1024 / 1024
        print(f"Descarga AEMET completada: {output_path.relative_to(config.project_root)} ({size_mb:.2f} MB)")
        return True


def get_aemet_api_key(environment_manager: EnvironmentManager) -> str | None:
    return environment_manager.read_env_variable("AEMET_API_KEY")


### 8.4 Descarga de fuentes públicas principales


In [ ]:
public_downloader = LocalFileDownloader(timeout_seconds=HTTP_TIMEOUT_SECONDS)

if not RUN_DOWNLOADS:
    print("RUN_DOWNLOADS está desactivado. No se descargan fuentes públicas.")
else:
    public_sources = [spec for spec in raw_dataset_specs if not spec.requires_api_key]

    for spec in public_sources:
        if spec.is_heavy_download and not RUN_HEAVY_DOWNLOADS:
            print(f"Saltando descarga pesada: {spec.dataset}")
            continue

        try:
            public_downloader.download(
                url=spec.source_url,
                output_path=spec.local_path,
                force=FORCE_REDOWNLOAD,
            )
        except Exception as error:
            print(f"No se pudo descargar {spec.dataset}: {error}")


### 8.5 Descarga de fuentes AEMET


In [ ]:
selected_aemet_station_ids = ["3126Y", "3129", "3195", "3196", "3200"]

# Se usan rangos mensuales para evitar problemas de límite de rango en AEMET
# y para hacer la descarga más recuperable si una petición falla.
aemet_daily_ranges_2022 = [
    ("2022-01-01T00:00:00UTC", "2022-01-31T23:59:59UTC"),
    ("2022-02-01T00:00:00UTC", "2022-02-28T23:59:59UTC"),
    ("2022-03-01T00:00:00UTC", "2022-03-31T23:59:59UTC"),
    ("2022-04-01T00:00:00UTC", "2022-04-30T23:59:59UTC"),
    ("2022-05-01T00:00:00UTC", "2022-05-31T23:59:59UTC"),
    ("2022-06-01T00:00:00UTC", "2022-06-30T23:59:59UTC"),
    ("2022-07-01T00:00:00UTC", "2022-07-31T23:59:59UTC"),
    ("2022-08-01T00:00:00UTC", "2022-08-31T23:59:59UTC"),
    ("2022-09-01T00:00:00UTC", "2022-09-30T23:59:59UTC"),
    ("2022-10-01T00:00:00UTC", "2022-10-31T23:59:59UTC"),
    ("2022-11-01T00:00:00UTC", "2022-11-30T23:59:59UTC"),
    ("2022-12-01T00:00:00UTC", "2022-12-31T23:59:59UTC"),
]


def json_file_has_records(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False

    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return False

    if isinstance(data, list):
        return len(data) > 0

    if isinstance(data, dict):
        return len(data) > 0

    return False


def download_aemet_endpoint_json_with_retries(
    client: AEMETOpenDataClient,
    endpoint: str,
    attempts: int = 3,
    wait_seconds: float = 3.0,
):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            return client.download_endpoint_json(endpoint)
        except Exception as error:
            last_error = error
            print(f"Intento {attempt}/{attempts} fallido: {error}")

            if attempt < attempts:
                time.sleep(wait_seconds)

    raise last_error


if not RUN_DOWNLOADS:
    print("RUN_DOWNLOADS está desactivado. No se descargan fuentes AEMET.")
else:
    aemet_api_key = get_aemet_api_key(environment_manager)

    if not aemet_api_key:
        print("No se ha encontrado AEMET_API_KEY. Las descargas de AEMET quedan pendientes.")
    else:
        aemet_client = AEMETOpenDataClient(
            api_key=aemet_api_key,
            timeout_seconds=HTTP_TIMEOUT_SECONDS,
            wait_seconds=max(AEMET_WAIT_SECONDS, 1.5),
        )

        inventory_endpoint = (
            "https://opendata.aemet.es/opendata/api/"
            "valores/climatologicos/inventarioestaciones/todasestaciones"
        )

        forecast_endpoint = (
            "https://opendata.aemet.es/opendata/api/"
            "prediccion/especifica/municipio/horaria/28079"
        )

        try:
            aemet_client.download_to_file(
                endpoint=inventory_endpoint,
                output_path=config.raw_weather_dir / "aemet_stations_inventory_raw.json",
                force=FORCE_REDOWNLOAD,
            )
        except Exception as error:
            print(f"No se pudo descargar el inventario de AEMET: {error}")

        try:
            aemet_client.download_to_file(
                endpoint=forecast_endpoint,
                output_path=config.raw_weather_dir / "aemet_forecast_hourly_madrid_raw.json",
                force=FORCE_REDOWNLOAD,
            )
        except Exception as error:
            print(f"No se pudo descargar la predicción horaria de AEMET: {error}")

        daily_climate_output_path = (
            config.raw_weather_dir / "aemet_daily_climate_selected_stations_2022_raw.json"
        )

        daily_file_is_valid = json_file_has_records(daily_climate_output_path)

        if daily_climate_output_path.exists() and not daily_file_is_valid:
            print(
                "El archivo AEMET diario existe, pero no contiene registros válidos. "
                "Se elimina para regenerarlo."
            )
            daily_climate_output_path.unlink()

        if daily_climate_output_path.exists() and not FORCE_REDOWNLOAD:
            print(f"Ya existe y contiene registros: {daily_climate_output_path.relative_to(config.project_root)}")
        else:
            all_daily_climate_records = []

            for station_id in selected_aemet_station_ids:
                for start_date, end_date in aemet_daily_ranges_2022:
                    encoded_start = quote(start_date, safe="")
                    encoded_end = quote(end_date, safe="")

                    endpoint = (
                        "https://opendata.aemet.es/opendata/api/"
                        f"valores/climatologicos/diarios/datos/fechaini/{encoded_start}/"
                        f"fechafin/{encoded_end}/estacion/{station_id}"
                    )

                    print(f"Descargando AEMET diario: estación={station_id}, rango={start_date} a {end_date}")

                    try:
                        records = download_aemet_endpoint_json_with_retries(
                            client=aemet_client,
                            endpoint=endpoint,
                            attempts=3,
                            wait_seconds=4.0,
                        )

                        if isinstance(records, list):
                            for record in records:
                                record["_station_id_requested"] = station_id
                                record["_range_start"] = start_date
                                record["_range_end"] = end_date

                            all_daily_climate_records.extend(records)

                        else:
                            all_daily_climate_records.append(
                                {
                                    "_station_id_requested": station_id,
                                    "_range_start": start_date,
                                    "_range_end": end_date,
                                    "_raw_response": records,
                                }
                            )

                    except Exception as error:
                        print(
                            "No se pudo descargar climatología diaria "
                            f"para estación={station_id}, rango={start_date} a {end_date}: {error}"
                        )

            daily_climate_output_path.parent.mkdir(parents=True, exist_ok=True)
            daily_climate_output_path.write_text(
                json.dumps(all_daily_climate_records, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )

            size_mb = daily_climate_output_path.stat().st_size / 1024 / 1024

            print(
                "Archivo AEMET diario generado: "
                f"{daily_climate_output_path.relative_to(config.project_root)} ({size_mb:.2f} MB)"
            )
            print(f"Registros guardados: {len(all_daily_climate_records)}")

            if len(all_daily_climate_records) == 0:
                print(
                    "Aviso: no se han guardado registros diarios de AEMET. "
                    "El notebook continuará, pero la base puede quedar sin meteorología histórica."
                )


### 8.6 Verificación posterior a la descarga


In [ ]:
raw_catalog_df = RawDatasetCatalog(raw_dataset_specs).to_dataframe()
display(raw_catalog_df)

total_raw_files = len(raw_catalog_df)
existing_raw_files = int(raw_catalog_df["Existe localmente"].sum())
missing_raw_files = total_raw_files - existing_raw_files

total_required_raw_files = int(raw_catalog_df["Necesario para base de modelado"].sum())
existing_required_raw_files = int(
    raw_catalog_df[
        raw_catalog_df["Necesario para base de modelado"]
    ]["Existe localmente"].sum()
)
missing_required_raw_files = total_required_raw_files - existing_required_raw_files

print("Estado general de archivos raw:")
print(f"Archivos raw esperados: {total_raw_files}")
print(f"Archivos raw existentes: {existing_raw_files}")
print(f"Archivos raw pendientes: {missing_raw_files}")

print("\nEstado de archivos raw necesarios para la base de modelado:")
print(f"Archivos necesarios esperados: {total_required_raw_files}")
print(f"Archivos necesarios existentes: {existing_required_raw_files}")
print(f"Archivos necesarios pendientes: {missing_required_raw_files}")

if missing_required_raw_files == 0:
    print("\nTodas las fuentes raw necesarias para la base de modelado existen localmente.")
else:
    print("\nAún faltan fuentes raw necesarias. Deben revisarse antes de limpiar.")


## 9. Validación inicial de datos raw

En esta sección se validan los archivos originales que existan localmente en `data/raw`.

La validación inicial no transforma los datos. Su objetivo es comprobar que los archivos descargados pueden leerse correctamente y que tienen un formato coherente con lo esperado.


In [ ]:
@dataclass
class RawFileValidationResult:
    dataset: str
    path: str
    expected_type: str
    exists: bool
    size_mb: float
    can_be_read: bool
    status: str
    details: str


class RawFileValidator:
    def __init__(self, spec: RawDatasetSpec, project_root: Path):
        self.spec = spec
        self.project_root = project_root
        self.path = spec.local_path

    def detect_expected_type(self) -> str:
        suffix = self.path.suffix.lower()
        if suffix == ".json":
            return "json"
        if suffix == ".csv":
            return "csv"
        if suffix == ".zip":
            return "zip"
        return "unknown"

    def validate(self) -> RawFileValidationResult:
        expected_type = self.detect_expected_type()

        if not self.path.exists():
            return RawFileValidationResult(
                dataset=self.spec.dataset,
                path=str(self.path.relative_to(self.project_root)),
                expected_type=expected_type,
                exists=False,
                size_mb=0,
                can_be_read=False,
                status="pendiente",
                details="El archivo no existe localmente.",
            )

        size_mb = round(self.path.stat().st_size / 1024 / 1024, 2)

        if size_mb == 0:
            return RawFileValidationResult(
                dataset=self.spec.dataset,
                path=str(self.path.relative_to(self.project_root)),
                expected_type=expected_type,
                exists=True,
                size_mb=size_mb,
                can_be_read=False,
                status="error",
                details="El archivo existe pero está vacío.",
            )

        try:
            if expected_type == "json":
                with self.path.open("r", encoding="utf-8") as file:
                    data = json.load(file)
                if isinstance(data, dict):
                    details = f"JSON válido tipo dict. Claves detectadas: {list(data.keys())[:10]}"
                elif isinstance(data, list):
                    details = f"JSON válido tipo list. Elementos detectados: {len(data)}"
                else:
                    details = f"JSON válido. Tipo raíz: {type(data).__name__}"

            elif expected_type == "csv":
                sample = pd.read_csv(self.path, sep=None, engine="python", nrows=5)
                details = f"CSV legible. Columnas detectadas: {list(sample.columns)}."

            elif expected_type == "zip":
                with zipfile.ZipFile(self.path, "r") as zip_file:
                    names = zip_file.namelist()
                details = f"ZIP legible. Archivos internos detectados: {len(names)}. Primeros: {names[:10]}"

            else:
                return RawFileValidationResult(
                    dataset=self.spec.dataset,
                    path=str(self.path.relative_to(self.project_root)),
                    expected_type=expected_type,
                    exists=True,
                    size_mb=size_mb,
                    can_be_read=False,
                    status="revisar",
                    details="Extensión no reconocida.",
                )

            return RawFileValidationResult(
                dataset=self.spec.dataset,
                path=str(self.path.relative_to(self.project_root)),
                expected_type=expected_type,
                exists=True,
                size_mb=size_mb,
                can_be_read=True,
                status="ok",
                details=details,
            )

        except Exception as error:
            return RawFileValidationResult(
                dataset=self.spec.dataset,
                path=str(self.path.relative_to(self.project_root)),
                expected_type=expected_type,
                exists=True,
                size_mb=size_mb,
                can_be_read=False,
                status="error",
                details=str(error),
            )


raw_validation_results = [
    RawFileValidator(spec=spec, project_root=config.project_root).validate()
    for spec in raw_dataset_specs
]

raw_validation_df = pd.DataFrame(
    [
        {
            "Dataset": result.dataset,
            "Ruta": result.path,
            "Tipo esperado": result.expected_type,
            "Existe": result.exists,
            "Tamaño MB": result.size_mb,
            "Se puede leer": result.can_be_read,
            "Estado": result.status,
            "Detalle": result.details,
        }
        for result in raw_validation_results
    ]
)

display(raw_validation_df)

validation_status_summary = (
    raw_validation_df
    .groupby("Estado", dropna=False)
    .agg(archivos=("Dataset", "count"), tamano_total_mb=("Tamaño MB", "sum"))
    .reset_index()
)
display(validation_status_summary)


## 10. Limpieza de datasets auxiliares

En esta sección se limpian datasets auxiliares del proyecto.

Estos datasets no sustituyen al histórico principal de estaciones BiciMAD 2022, pero ayudan a completar el contexto del proyecto.

### 10.1 GBFS estaciones actuales


In [ ]:
def safe_read_json(path: Path):
    if not path.exists():
        print(f"No existe: {path.relative_to(config.project_root)}")
        return None
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def extract_gbfs_stations(raw_data: dict) -> list[dict]:
    if raw_data is None:
        return []
    if isinstance(raw_data, dict) and "data" in raw_data and isinstance(raw_data["data"], dict):
        if "stations" in raw_data["data"]:
            return raw_data["data"]["stations"]
    if isinstance(raw_data, dict) and "stations" in raw_data:
        return raw_data["stations"]
    return []


station_information_raw_path = config.raw_bicimad_dir / "gbfs_station_information_raw.json"
station_information_output_path = config.interim_bicimad_dir / "stations_clean.csv"

station_information_raw = safe_read_json(station_information_raw_path)
station_information_records = extract_gbfs_stations(station_information_raw)

if not station_information_records:
    stations_clean_df = pd.DataFrame()
    print("No hay registros de estaciones actuales para limpiar.")
else:
    df = pd.json_normalize(station_information_records)

    rename_map = {
        "station_id": "station_id_current",
        "name": "station_name",
        "lat": "station_latitude",
        "lon": "station_longitude",
        "capacity": "station_capacity",
        "address": "station_address",
    }
    df = df.rename(columns=rename_map)

    expected_columns = [
        "station_id_current",
        "station_name",
        "station_latitude",
        "station_longitude",
        "station_capacity",
        "station_address",
    ]

    for column in expected_columns:
        if column not in df.columns:
            df[column] = pd.NA

    stations_clean_df = df[expected_columns].copy()

    for column in ["station_latitude", "station_longitude", "station_capacity"]:
        stations_clean_df[column] = pd.to_numeric(stations_clean_df[column], errors="coerce")

    stations_clean_df["station_id_current"] = stations_clean_df["station_id_current"].astype("string")
    stations_clean_df["station_name"] = stations_clean_df["station_name"].astype("string")
    stations_clean_df["station_address"] = stations_clean_df["station_address"].astype("string")

    stations_clean_df = stations_clean_df.sort_values("station_id_current").reset_index(drop=True)
    stations_clean_df.to_csv(station_information_output_path, index=False)

    print(f"Archivo generado: {station_information_output_path.relative_to(config.project_root)}")
    print(f"Filas: {len(stations_clean_df)}")
    display(stations_clean_df.head())


### 10.2 GBFS estado actual


In [ ]:
station_status_raw_path = config.raw_bicimad_dir / "gbfs_station_status_snapshot_raw.json"
station_status_output_path = config.interim_bicimad_dir / "station_status_snapshot_clean.csv"

station_status_raw = safe_read_json(station_status_raw_path)
station_status_records = extract_gbfs_stations(station_status_raw)

if not station_status_records:
    station_status_snapshot_clean_df = pd.DataFrame()
    print("No hay registros de estado actual para limpiar.")
else:
    df = pd.json_normalize(station_status_records)

    expected_columns = [
        "station_id",
        "num_bikes_available",
        "num_docks_available",
        "num_bikes_disabled",
        "num_docks_disabled",
        "is_installed",
        "is_renting",
        "is_returning",
        "last_reported",
    ]

    for column in expected_columns:
        if column not in df.columns:
            df[column] = pd.NA

    station_status_snapshot_clean_df = df[expected_columns].copy()
    station_status_snapshot_clean_df = station_status_snapshot_clean_df.rename(
        columns={"station_id": "station_id_current"}
    )

    station_status_snapshot_clean_df["station_id_current"] = (
        station_status_snapshot_clean_df["station_id_current"].astype("string")
    )

    numeric_columns = [
        "num_bikes_available",
        "num_docks_available",
        "num_bikes_disabled",
        "num_docks_disabled",
        "is_installed",
        "is_renting",
        "is_returning",
        "last_reported",
    ]

    for column in numeric_columns:
        station_status_snapshot_clean_df[column] = pd.to_numeric(
            station_status_snapshot_clean_df[column],
            errors="coerce",
        )

    station_status_snapshot_clean_df["last_reported_datetime"] = pd.to_datetime(
        station_status_snapshot_clean_df["last_reported"],
        unit="s",
        errors="coerce",
    )

    station_status_snapshot_clean_df["station_available_for_renting_and_returning"] = (
        (station_status_snapshot_clean_df["is_installed"] == 1)
        & (station_status_snapshot_clean_df["is_renting"] == 1)
        & (station_status_snapshot_clean_df["is_returning"] == 1)
    )

    station_status_snapshot_clean_df = station_status_snapshot_clean_df.sort_values("station_id_current").reset_index(drop=True)
    station_status_snapshot_clean_df.to_csv(station_status_output_path, index=False)

    print(f"Archivo generado: {station_status_output_path.relative_to(config.project_root)}")
    print(f"Filas: {len(station_status_snapshot_clean_df)}")
    display(station_status_snapshot_clean_df.head())


### 10.3 Histórico diario de viajes


In [ ]:
daily_trips_raw_path = config.raw_bicimad_dir / "bicimad_daily_trips_raw.csv"
daily_trips_output_path = config.interim_bicimad_dir / "bicimad_daily_trips_clean.csv"

if not daily_trips_raw_path.exists():
    daily_trips_clean_df = pd.DataFrame()
    print(f"No existe: {daily_trips_raw_path.relative_to(config.project_root)}")
else:
    daily_trips_raw_df = pd.read_csv(daily_trips_raw_path, sep=None, engine="python")
    daily_trips_raw_df.columns = [str(col).strip().upper() for col in daily_trips_raw_df.columns]

    date_col = "FECHA" if "FECHA" in daily_trips_raw_df.columns else daily_trips_raw_df.columns[0]
    trips_col = "VIAJES" if "VIAJES" in daily_trips_raw_df.columns else daily_trips_raw_df.columns[1]

    daily_trips_clean_df = daily_trips_raw_df[[date_col, trips_col]].copy()
    daily_trips_clean_df = daily_trips_clean_df.rename(
        columns={date_col: "date", trips_col: "daily_trips"}
    )

    daily_trips_clean_df["date"] = pd.to_datetime(
        daily_trips_clean_df["date"].astype(str),
        format="%Y%m%d",
        errors="coerce",
    ).fillna(
        pd.to_datetime(daily_trips_clean_df["date"], errors="coerce", dayfirst=True)
    )

    daily_trips_clean_df["daily_trips"] = pd.to_numeric(
        daily_trips_clean_df["daily_trips"],
        errors="coerce",
    )

    daily_trips_clean_df = (
        daily_trips_clean_df
        .dropna(subset=["date"])
        .drop_duplicates(subset=["date"])
        .sort_values("date")
        .reset_index(drop=True)
    )

    daily_trips_clean_df.to_csv(daily_trips_output_path, index=False)

    print(f"Archivo generado: {daily_trips_output_path.relative_to(config.project_root)}")
    print(f"Rango de fechas: {daily_trips_clean_df['date'].min()} → {daily_trips_clean_df['date'].max()}")
    print(f"Filas: {len(daily_trips_clean_df)}")
    display(daily_trips_clean_df.head())


### 10.4 Calendario laboral


In [ ]:
calendar_raw_path = config.raw_calendar_dir / "madrid_working_calendar_raw.csv"
calendar_output_path = config.interim_calendar_dir / "madrid_working_calendar_clean.csv"

def normalize_colname(value: str) -> str:
    value = str(value).strip().lower()
    replacements = {
        "á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u",
        "ñ": "n", " ": "_", "-": "_", ".": "_"
    }
    for old, new in replacements.items():
        value = value.replace(old, new)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")


def find_best_date_column(df: pd.DataFrame) -> str:
    best_column = None
    best_score = -1

    for column in df.columns:
        parsed = pd.to_datetime(df[column], errors="coerce", dayfirst=True)
        score = parsed.notna().sum()

        if score > best_score:
            best_score = score
            best_column = column

    if best_column is None or best_score == 0:
        raise ValueError("No se ha podido detectar una columna de fecha en el calendario.")

    return best_column


if not calendar_raw_path.exists():
    calendar_clean_df = pd.DataFrame()
    print(f"No existe: {calendar_raw_path.relative_to(config.project_root)}")
else:
    calendar_raw_df = pd.read_csv(
        calendar_raw_path,
        sep=None,
        engine="python",
        encoding="utf-8-sig",
    )

    calendar_raw_df.columns = [normalize_colname(col) for col in calendar_raw_df.columns]

    date_col = find_best_date_column(calendar_raw_df)

    calendar_clean_df = pd.DataFrame()
    calendar_clean_df["date"] = pd.to_datetime(
        calendar_raw_df[date_col],
        errors="coerce",
        dayfirst=True,
    )

    calendar_clean_df = calendar_clean_df.dropna(subset=["date"]).copy()

    text_context = (
        calendar_raw_df
        .reindex(calendar_clean_df.index)
        .astype("string")
        .fillna("")
        .apply(lambda row: " ".join([str(value) for value in row.to_list()]).lower(), axis=1)
    )

    day_name_map_es = {
        0: "lunes",
        1: "martes",
        2: "miércoles",
        3: "jueves",
        4: "viernes",
        5: "sábado",
        6: "domingo",
    }

    calendar_clean_df["year"] = calendar_clean_df["date"].dt.year
    calendar_clean_df["month"] = calendar_clean_df["date"].dt.month
    calendar_clean_df["day_of_week"] = calendar_clean_df["date"].dt.dayofweek
    calendar_clean_df["day_of_week_name_es"] = calendar_clean_df["day_of_week"].map(day_name_map_es)

    calendar_clean_df["is_weekend"] = calendar_clean_df["day_of_week"].isin([5, 6])
    calendar_clean_df["is_holiday"] = text_context.str.contains("festivo|fiesta|inhabil", regex=True, na=False)

    calendar_clean_df["day_type"] = np.select(
        [
            calendar_clean_df["is_holiday"],
            calendar_clean_df["day_of_week"] == 5,
            calendar_clean_df["day_of_week"] == 6,
        ],
        [
            "festivo",
            "sabado",
            "domingo",
        ],
        default="laborable",
    )

    calendar_clean_df["is_working_day"] = calendar_clean_df["day_type"].eq("laborable")
    calendar_clean_df["is_non_working_day"] = ~calendar_clean_df["is_working_day"]
    calendar_clean_df["holiday_type"] = np.where(calendar_clean_df["is_holiday"], "festivo", pd.NA)
    calendar_clean_df["holiday_name"] = pd.NA

    calendar_clean_df = (
        calendar_clean_df
        .drop_duplicates(subset=["date"])
        .sort_values("date")
        .reset_index(drop=True)
    )

    calendar_clean_df.to_csv(calendar_output_path, index=False)

    print(f"Archivo generado: {calendar_output_path.relative_to(config.project_root)}")
    print(f"Rango de fechas: {calendar_clean_df['date'].min()} → {calendar_clean_df['date'].max()}")
    print(f"Filas: {len(calendar_clean_df)}")
    display(calendar_clean_df.head())


### 10.5 Meteorología AEMET


In [ ]:
def to_float_es(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().replace(",", ".")

    if text in {"", "nan", "None"}:
        return np.nan

    if text.lower() in {"ip", "tr", "trace"}:
        return 0.0

    match = re.search(r"-?\d+(?:\.\d+)?", text)
    if not match:
        return np.nan

    return float(match.group(0))


# Inventario AEMET
aemet_inventory_raw_path = config.raw_weather_dir / "aemet_stations_inventory_raw.json"
aemet_inventory_output_path = config.interim_weather_dir / "aemet_stations_inventory_clean.csv"

inventory_raw = safe_read_json(aemet_inventory_raw_path)

if isinstance(inventory_raw, list):
    inventory_df = pd.DataFrame(inventory_raw)
elif isinstance(inventory_raw, dict):
    inventory_df = pd.json_normalize(inventory_raw)
else:
    inventory_df = pd.DataFrame()

if inventory_df.empty:
    aemet_inventory_clean_df = pd.DataFrame()
    print("No hay inventario AEMET para limpiar.")
else:
    aemet_inventory_clean_df = inventory_df.copy()
    aemet_inventory_clean_df.columns = [normalize_colname(col) for col in aemet_inventory_clean_df.columns]
    aemet_inventory_clean_df.to_csv(aemet_inventory_output_path, index=False)
    print(f"Archivo generado: {aemet_inventory_output_path.relative_to(config.project_root)}")
    display(aemet_inventory_clean_df.head())


# Climatología diaria AEMET 2022
aemet_daily_raw_path = config.raw_weather_dir / "aemet_daily_climate_selected_stations_2022_raw.json"
aemet_daily_output_path = config.interim_weather_dir / "aemet_daily_climate_selected_stations_2022_clean.csv"
aemet_daily_join_ready_path = config.interim_weather_dir / "aemet_daily_weather_madrid_2022_join_ready.csv"

daily_weather_raw = safe_read_json(aemet_daily_raw_path)

if isinstance(daily_weather_raw, list):
    daily_weather_df = pd.DataFrame(daily_weather_raw)
else:
    daily_weather_df = pd.DataFrame()

if daily_weather_df.empty:
    aemet_daily_weather_madrid_2022_join_ready_df = pd.DataFrame()
    print("No hay datos diarios AEMET para limpiar.")
else:
    daily_weather_df.columns = [normalize_colname(col) for col in daily_weather_df.columns]

    date_col = "fecha" if "fecha" in daily_weather_df.columns else None
    station_col = "indicativo" if "indicativo" in daily_weather_df.columns else "_station_id_requested"

    if date_col is None:
        raise ValueError("No se ha encontrado columna de fecha en AEMET diario.")

    aemet_daily_clean_df = pd.DataFrame()
    aemet_daily_clean_df["date"] = pd.to_datetime(daily_weather_df[date_col], errors="coerce")
    aemet_daily_clean_df["weather_station_id"] = daily_weather_df[station_col].astype("string")

    source_map = {
        "weather_temperature_mean_c": ["tmed", "temperatura_media"],
        "weather_temperature_min_c": ["tmin", "temperatura_minima"],
        "weather_temperature_max_c": ["tmax", "temperatura_maxima"],
        "weather_precipitation_mm": ["prec", "precipitacion"],
        "weather_humidity_mean_pct": ["hrmedia", "humedad_media"],
    }

    for target_col, possible_cols in source_map.items():
        source_col = next((col for col in possible_cols if col in daily_weather_df.columns), None)
        if source_col is None:
            aemet_daily_clean_df[target_col] = np.nan
        else:
            aemet_daily_clean_df[target_col] = daily_weather_df[source_col].apply(to_float_es)

    aemet_daily_clean_df = aemet_daily_clean_df.dropna(subset=["date"]).copy()
    aemet_daily_clean_df["weather_has_precipitation"] = aemet_daily_clean_df["weather_precipitation_mm"].fillna(0) > 0

    aemet_daily_clean_df.to_csv(aemet_daily_output_path, index=False)

    aemet_daily_weather_madrid_2022_join_ready_df = (
        aemet_daily_clean_df
        .groupby("date", as_index=False)
        .agg(
            weather_temperature_mean_c=("weather_temperature_mean_c", "mean"),
            weather_temperature_min_c=("weather_temperature_min_c", "mean"),
            weather_temperature_max_c=("weather_temperature_max_c", "mean"),
            weather_precipitation_mm=("weather_precipitation_mm", "mean"),
            weather_humidity_mean_pct=("weather_humidity_mean_pct", "mean"),
            weather_station_count=("weather_station_id", "nunique"),
            weather_temperature_available_count=("weather_temperature_mean_c", lambda s: int(s.notna().sum())),
            weather_precipitation_available_count=("weather_precipitation_mm", lambda s: int(s.notna().sum())),
            weather_humidity_available_count=("weather_humidity_mean_pct", lambda s: int(s.notna().sum())),
        )
    )

    aemet_daily_weather_madrid_2022_join_ready_df["weather_has_precipitation"] = (
        aemet_daily_weather_madrid_2022_join_ready_df["weather_precipitation_mm"].fillna(0) > 0
    )

    aemet_daily_weather_madrid_2022_join_ready_df = (
        aemet_daily_weather_madrid_2022_join_ready_df
        .sort_values("date")
        .reset_index(drop=True)
    )

    aemet_daily_weather_madrid_2022_join_ready_df.to_csv(aemet_daily_join_ready_path, index=False)

    print(f"Archivo generado: {aemet_daily_output_path.relative_to(config.project_root)}")
    print(f"Archivo generado: {aemet_daily_join_ready_path.relative_to(config.project_root)}")
    print(f"Rango de fechas: {aemet_daily_weather_madrid_2022_join_ready_df['date'].min()} → {aemet_daily_weather_madrid_2022_join_ready_df['date'].max()}")
    display(aemet_daily_weather_madrid_2022_join_ready_df.head())


# Predicción horaria AEMET para demo futura
forecast_raw_path = config.raw_weather_dir / "aemet_forecast_hourly_madrid_raw.json"
forecast_output_path = config.interim_weather_dir / "aemet_forecast_hourly_madrid_clean.csv"
forecast_raw = safe_read_json(forecast_raw_path)

forecast_records = []

try:
    if isinstance(forecast_raw, list) and forecast_raw:
        prediccion = forecast_raw[0].get("prediccion", {})
        dias = prediccion.get("dia", [])

        for dia in dias:
            fecha = dia.get("fecha")
            for variable_group in ["temperatura", "humedadRelativa", "precipitacion"]:
                for item in dia.get(variable_group, []):
                    forecast_records.append(
                        {
                            "forecast_date": fecha,
                            "hour": item.get("periodo"),
                            "variable": variable_group,
                            "value": item.get("value"),
                        }
                    )

    aemet_forecast_hourly_madrid_clean_df = pd.DataFrame(forecast_records)

    if not aemet_forecast_hourly_madrid_clean_df.empty:
        aemet_forecast_hourly_madrid_clean_df.to_csv(forecast_output_path, index=False)
        print(f"Archivo generado: {forecast_output_path.relative_to(config.project_root)}")
        display(aemet_forecast_hourly_madrid_clean_df.head())
    else:
        print("No se han encontrado registros horarios en la predicción AEMET.")
except Exception as error:
    print(f"No se pudo limpiar la predicción horaria AEMET: {error}")


## 11. Limpieza del histórico principal BiciMAD 2022

Esta sección limpia el histórico de estado de estaciones de BiciMAD 2022.

Es la fuente principal del proyecto. El resultado será:

`data/interim/bicimad/station_status_history_2022_clean.csv`

La limpieza se ha diseñado para ser robusta ante distintas estructuras internas del ZIP histórico.


In [ ]:
historical_raw_zip_path = config.raw_bicimad_dir / "bicimad_station_status_history_2022_raw.zip"
historical_clean_output_path = config.interim_bicimad_dir / "station_status_history_2022_clean.csv"
historical_zip_inventory_debug_path = config.interim_bicimad_dir / "historical_zip_inventory_debug.csv"

historical_output_columns = [
    "source_month",
    "snapshot_datetime",
    "station_id_historical",
    "station_number",
    "station_name",
    "station_address",
    "station_latitude",
    "station_longitude",
    "station_capacity",
    "num_docks_available",
    "num_bikes_available",
    "reservations_count",
    "station_light_status",
    "is_active",
    "is_not_available",
    "occupation_ratio",
]


def safe_decode_json_bytes(raw_bytes: bytes):
    for encoding in ["utf-8", "utf-8-sig", "latin1"]:
        try:
            text = raw_bytes.decode(encoding)
            break
        except UnicodeDecodeError:
            text = None

    if text is None:
        return None

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        records = []

        for line in text.splitlines():
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue

        if records:
            return records

    return None


def iter_files_recursively_from_zip_bytes(raw_zip_bytes: bytes, prefix: str = ""):
    try:
        with zipfile.ZipFile(BytesIO(raw_zip_bytes), "r") as zip_file:
            for zip_info in zip_file.infolist():
                if zip_info.is_dir():
                    continue

                file_name = zip_info.filename
                full_name = f"{prefix}/{file_name}" if prefix else file_name
                lower_name = file_name.lower()

                try:
                    file_bytes = zip_file.read(file_name)
                except Exception:
                    continue

                if lower_name.endswith(".zip"):
                    yield from iter_files_recursively_from_zip_bytes(file_bytes, prefix=full_name)
                else:
                    yield full_name, zip_info, file_bytes

    except zipfile.BadZipFile:
        return


def iter_files_recursively_from_zip_path(zip_path: Path):
    with zip_path.open("rb") as file:
        raw_zip_bytes = file.read()

    yield from iter_files_recursively_from_zip_bytes(raw_zip_bytes)


def get_nested_value(record: dict, keys: list[str], default=np.nan):
    for key in keys:
        if "." in key:
            current = record
            ok = True

            for part in key.split("."):
                if isinstance(current, dict) and part in current:
                    current = current[part]
                else:
                    ok = False
                    break

            if ok:
                return current

        elif isinstance(record, dict) and key in record:
            return record[key]

    return default


def extract_coordinates(record: dict):
    lat = get_nested_value(record, ["lat", "latitude", "station_latitude"], np.nan)
    lon = get_nested_value(record, ["lon", "lng", "longitude", "station_longitude"], np.nan)

    geometry = record.get("geometry") if isinstance(record, dict) else None

    if isinstance(geometry, dict):
        coords = geometry.get("coordinates")

        if isinstance(coords, list) and len(coords) >= 2:
            lon = coords[0]
            lat = coords[1]

    return lat, lon


def parse_snapshot_datetime(root, file_name: str, zip_info=None):
    candidates = []

    if isinstance(root, dict):
        candidates.extend(
            [
                root.get("last_updated"),
                root.get("datetime"),
                root.get("timestamp"),
                root.get("date"),
                root.get("fecha"),
            ]
        )

        if isinstance(root.get("data"), dict):
            data = root["data"]
            candidates.extend(
                [
                    data.get("last_updated"),
                    data.get("datetime"),
                    data.get("timestamp"),
                    data.get("date"),
                    data.get("fecha"),
                ]
            )

    for value in candidates:
        if value is None:
            continue

        if isinstance(value, (int, float)) or str(value).isdigit():
            parsed = pd.to_datetime(value, unit="s", errors="coerce")
        else:
            parsed = pd.to_datetime(value, errors="coerce")

        if pd.notna(parsed):
            return parsed

    patterns = [
        r"(20\d{2})[-_]?([01]\d)[-_]?([0-3]\d).*?([0-2]\d)[-_:]?([0-5]\d)(?:[-_:]?([0-5]\d))?",
        r"(20\d{2})[-_]?([01]\d)[-_]?([0-3]\d)",
    ]

    for pattern in patterns:
        match = re.search(pattern, file_name)

        if not match:
            continue

        groups = match.groups()

        if len(groups) >= 6:
            year, month, day, hour, minute, second = groups
            second = second or "00"
            parsed = pd.to_datetime(f"{year}-{month}-{day} {hour}:{minute}:{second}", errors="coerce")
        else:
            year, month, day = groups[:3]
            parsed = pd.to_datetime(f"{year}-{month}-{day}", errors="coerce")

        if pd.notna(parsed):
            return parsed

    if zip_info is not None:
        try:
            year, month, day, hour, minute, second = zip_info.date_time
            return pd.Timestamp(year, month, day, hour, minute, second)
        except Exception:
            pass

    return pd.NaT


def month_from_name(name: str) -> str:
    match = re.search(r"(20\d{2})[-_]?([01]\d)", name)

    if match:
        return f"{match.group(1)}-{match.group(2)}"

    return "unknown"


def station_candidate_score(record: dict) -> int:
    if not isinstance(record, dict):
        return 0

    station_keys = {
        "id",
        "station_id",
        "number",
        "name",
        "address",
        "geometry",
        "dock_bikes",
        "free_bases",
        "total_bases",
        "reservations_count",
        "light",
        "activate",
        "no_available",
        "num_bikes_available",
        "num_docks_available",
        "capacity",
    }

    return len(station_keys.intersection(set(record.keys())))


def looks_like_station_status_records(records) -> bool:
    if not isinstance(records, list) or not records:
        return False

    sample = [record for record in records[:20] if isinstance(record, dict)]

    if not sample:
        return False

    score = sum(station_candidate_score(record) for record in sample) / len(sample)

    availability_keys = {
        "dock_bikes",
        "free_bases",
        "num_bikes_available",
        "num_docks_available",
        "total_bases",
        "capacity",
    }

    has_availability = any(
        bool(availability_keys.intersection(set(record.keys())))
        for record in sample
    )

    has_station_identity = any(
        {"id", "station_id", "number", "name"}.intersection(set(record.keys()))
        for record in sample
    )

    return score >= 3 and has_availability and has_station_identity


def find_station_status_lists(payload):
    found_lists = []

    def walk(obj):
        if isinstance(obj, list):
            if looks_like_station_status_records(obj):
                found_lists.append(obj)
                return

            for item in obj:
                walk(item)

        elif isinstance(obj, dict):
            for value in obj.values():
                walk(value)

    walk(payload)

    return found_lists


def to_numeric_or_nan(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().replace(",", ".")

    if text == "" or text.lower() in {"nan", "none", "null"}:
        return np.nan

    match = re.search(r"-?\d+(?:\.\d+)?", text)

    if not match:
        return np.nan

    return float(match.group(0))


def to_int_flag(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, bool):
        return int(value)

    text = str(value).strip().lower()

    if text in {"1", "true", "t", "yes", "y", "si", "sí"}:
        return 1

    if text in {"0", "false", "f", "no", "n"}:
        return 0

    try:
        return int(float(text))
    except Exception:
        return np.nan


def normalize_historical_station_record(record: dict, snapshot_datetime, source_month: str):
    lat, lon = extract_coordinates(record)

    station_capacity = get_nested_value(
        record,
        ["capacity", "total_bases", "station_capacity"],
        np.nan,
    )

    num_bikes_available = get_nested_value(
        record,
        ["num_bikes_available", "dock_bikes", "bikes_available"],
        np.nan,
    )

    num_docks_available = get_nested_value(
        record,
        ["num_docks_available", "free_bases", "docks_available"],
        np.nan,
    )

    station_light_status = get_nested_value(
        record,
        ["light", "station_light_status"],
        np.nan,
    )

    is_active = get_nested_value(
        record,
        ["activate", "is_active", "is_installed"],
        np.nan,
    )

    is_not_available = get_nested_value(
        record,
        ["no_available", "is_not_available"],
        np.nan,
    )

    station_light_status = to_numeric_or_nan(station_light_status)
    is_not_available = to_int_flag(is_not_available)

    if pd.isna(is_not_available) and station_light_status == 3:
        is_not_available = 1

    return {
        "source_month": source_month,
        "snapshot_datetime": snapshot_datetime,
        "station_id_historical": get_nested_value(record, ["id", "station_id", "station_id_historical"], np.nan),
        "station_number": get_nested_value(record, ["number", "station_number"], np.nan),
        "station_name": get_nested_value(record, ["name", "station_name"], pd.NA),
        "station_address": get_nested_value(record, ["address", "station_address"], pd.NA),
        "station_latitude": lat,
        "station_longitude": lon,
        "station_capacity": station_capacity,
        "num_docks_available": num_docks_available,
        "num_bikes_available": num_bikes_available,
        "reservations_count": get_nested_value(record, ["reservations_count", "reservation_count"], np.nan),
        "station_light_status": station_light_status,
        "is_active": to_int_flag(is_active),
        "is_not_available": is_not_available,
    }


if not historical_raw_zip_path.exists():
    station_status_history_2022_clean_df = pd.DataFrame(columns=historical_output_columns)
    print(f"No existe: {historical_raw_zip_path.relative_to(config.project_root)}")
else:
    print("Procesando ZIP histórico. Esta celda puede tardar varios minutos.")

    historical_clean_output_path.parent.mkdir(parents=True, exist_ok=True)

    if historical_clean_output_path.exists():
        historical_clean_output_path.unlink()

    inventory_records = []
    buffer = []
    total_records = 0
    processed_json_files = 0
    json_files_seen = 0
    written_header = False
    chunk_size = 100_000

    for file_name, zip_info, raw_bytes in iter_files_recursively_from_zip_path(historical_raw_zip_path):
        lower_name = file_name.lower()

        inventory_records.append(
            {
                "file_name": file_name,
                "size_bytes": len(raw_bytes),
                "is_json": lower_name.endswith(".json"),
            }
        )

        if not lower_name.endswith(".json"):
            continue

        json_files_seen += 1
        payload = safe_decode_json_bytes(raw_bytes)

        if payload is None:
            continue

        station_lists = find_station_status_lists(payload)

        if not station_lists:
            continue

        snapshot_datetime = parse_snapshot_datetime(payload, file_name, zip_info=zip_info)
        source_month = month_from_name(file_name)

        for station_records in station_lists:
            processed_json_files += 1

            for record in station_records:
                normalized = normalize_historical_station_record(
                    record=record,
                    snapshot_datetime=snapshot_datetime,
                    source_month=source_month,
                )
                buffer.append(normalized)

        if len(buffer) >= chunk_size:
            chunk_df = pd.DataFrame(buffer)

            for column in historical_output_columns:
                if column not in chunk_df.columns:
                    chunk_df[column] = pd.NA

            chunk_df = chunk_df[historical_output_columns[:-1]].copy()

            chunk_df["station_capacity"] = pd.to_numeric(chunk_df["station_capacity"], errors="coerce")
            chunk_df["num_bikes_available"] = pd.to_numeric(chunk_df["num_bikes_available"], errors="coerce")
            chunk_df["occupation_ratio"] = (
                chunk_df["num_bikes_available"]
                / chunk_df["station_capacity"].replace({0: np.nan})
            ).clip(0, 1)

            chunk_df = chunk_df[historical_output_columns]

            chunk_df.to_csv(
                historical_clean_output_path,
                index=False,
                mode="a",
                header=not written_header,
            )

            written_header = True
            total_records += len(chunk_df)
            buffer = []

            print(f"Registros escritos: {total_records:,}")

    inventory_df = pd.DataFrame(inventory_records)
    inventory_df.to_csv(historical_zip_inventory_debug_path, index=False)

    if buffer:
        chunk_df = pd.DataFrame(buffer)

        for column in historical_output_columns:
            if column not in chunk_df.columns:
                chunk_df[column] = pd.NA

        chunk_df = chunk_df[historical_output_columns[:-1]].copy()

        chunk_df["station_capacity"] = pd.to_numeric(chunk_df["station_capacity"], errors="coerce")
        chunk_df["num_bikes_available"] = pd.to_numeric(chunk_df["num_bikes_available"], errors="coerce")
        chunk_df["occupation_ratio"] = (
            chunk_df["num_bikes_available"]
            / chunk_df["station_capacity"].replace({0: np.nan})
        ).clip(0, 1)

        chunk_df = chunk_df[historical_output_columns]

        chunk_df.to_csv(
            historical_clean_output_path,
            index=False,
            mode="a",
            header=not written_header,
        )

        written_header = True
        total_records += len(chunk_df)

    if total_records == 0:
        pd.DataFrame(columns=historical_output_columns).to_csv(historical_clean_output_path, index=False)

    print(f"Archivos internos inspeccionados: {len(inventory_records):,}")
    print(f"Archivos JSON encontrados: {json_files_seen:,}")
    print(f"Listas JSON con estructura de estación detectadas: {processed_json_files:,}")
    print(f"Registros históricos escritos: {total_records:,}")
    print(f"Inventario debug generado: {historical_zip_inventory_debug_path.relative_to(config.project_root)}")
    print(f"Archivo generado: {historical_clean_output_path.relative_to(config.project_root)}")

    station_status_history_2022_clean_df = pd.read_csv(
        historical_clean_output_path,
        parse_dates=["snapshot_datetime"],
        low_memory=False,
    )

    numeric_columns = [
        "station_id_historical",
        "station_number",
        "station_latitude",
        "station_longitude",
        "station_capacity",
        "num_docks_available",
        "num_bikes_available",
        "reservations_count",
        "station_light_status",
        "is_active",
        "is_not_available",
        "occupation_ratio",
    ]

    for column in numeric_columns:
        if column in station_status_history_2022_clean_df.columns:
            station_status_history_2022_clean_df[column] = pd.to_numeric(
                station_status_history_2022_clean_df[column],
                errors="coerce",
            )

    print("Validación rápida del histórico limpio:")
    print(f"Filas: {len(station_status_history_2022_clean_df):,}")
    print(f"Columnas: {len(station_status_history_2022_clean_df.columns)}")

    if not station_status_history_2022_clean_df.empty:
        print(f"Fecha mínima: {station_status_history_2022_clean_df['snapshot_datetime'].min()}")
        print(f"Fecha máxima: {station_status_history_2022_clean_df['snapshot_datetime'].max()}")
        print(f"Estaciones únicas: {station_status_history_2022_clean_df['station_id_historical'].nunique()}")
        display(station_status_history_2022_clean_df.head())
    else:
        print(
            "No se han extraído registros de estaciones. "
            "Revisa el inventario debug para ver la estructura interna del ZIP."
        )
        display(inventory_df.head(30))


## 12. Creación de la base enriquecida intermedia

Esta sección une el histórico limpio de BiciMAD 2022 con calendario laboral y meteorología diaria.

El resultado es:

`data/interim/bicimad/station_status_history_2022_enriched_base.csv`


In [ ]:
enriched_base_output_path = config.interim_bicimad_dir / "station_status_history_2022_enriched_base.csv"
processed_modeling_base_output_path = config.processed_dir / "station_status_history_2022_modeling_base.csv"


def parse_datetime_safely(series: pd.Series) -> pd.Series:
    try:
        return pd.to_datetime(series, errors="coerce", format="mixed")
    except TypeError:
        return pd.to_datetime(series, errors="coerce")


if historical_clean_output_path.exists():
    station_history_df = pd.read_csv(
        historical_clean_output_path,
        low_memory=False,
    )
else:
    station_history_df = pd.DataFrame()

calendar_clean_path = config.interim_calendar_dir / "madrid_working_calendar_clean.csv"
weather_join_ready_path = config.interim_weather_dir / "aemet_daily_weather_madrid_2022_join_ready.csv"

calendar_for_join_df = (
    pd.read_csv(calendar_clean_path)
    if calendar_clean_path.exists()
    else pd.DataFrame()
)

weather_for_join_df = (
    pd.read_csv(weather_join_ready_path)
    if weather_join_ready_path.exists()
    else pd.DataFrame()
)

if station_history_df.empty:
    enriched_base_df = pd.DataFrame()
    print("No se puede crear la base enriquecida porque falta el histórico limpio.")
else:
    enriched_base_df = station_history_df.copy()

    enriched_base_df["snapshot_datetime"] = parse_datetime_safely(
        enriched_base_df["snapshot_datetime"]
    )

    missing_snapshot_datetime = int(enriched_base_df["snapshot_datetime"].isna().sum())

    if missing_snapshot_datetime > 0:
        print(
            "Aviso: hay registros sin snapshot_datetime válido. "
            f"Registros afectados: {missing_snapshot_datetime:,}"
        )

    enriched_base_df["date"] = enriched_base_df["snapshot_datetime"].dt.normalize()
    enriched_base_df["snapshot_hour"] = enriched_base_df["snapshot_datetime"].dt.hour
    enriched_base_df["snapshot_day_of_week"] = enriched_base_df["snapshot_datetime"].dt.dayofweek
    enriched_base_df["snapshot_month"] = enriched_base_df["snapshot_datetime"].dt.month

    if not calendar_for_join_df.empty:
        calendar_for_join_df["date"] = parse_datetime_safely(calendar_for_join_df["date"]).dt.normalize()

        calendar_cols = [
            "date",
            "year",
            "month",
            "day_of_week",
            "day_of_week_name_es",
            "day_type",
            "is_working_day",
            "is_non_working_day",
            "is_weekend",
            "is_holiday",
            "holiday_type",
            "holiday_name",
        ]

        available_calendar_cols = [
            col for col in calendar_cols
            if col in calendar_for_join_df.columns
        ]

        enriched_base_df = enriched_base_df.merge(
            calendar_for_join_df[available_calendar_cols],
            on="date",
            how="left",
        )
    else:
        print("Calendario limpio no disponible. Se crearán variables temporales básicas.")

        enriched_base_df["day_type"] = np.select(
            [
                enriched_base_df["snapshot_day_of_week"] == 5,
                enriched_base_df["snapshot_day_of_week"] == 6,
            ],
            ["sabado", "domingo"],
            default="laborable",
        )

        enriched_base_df["is_weekend"] = enriched_base_df["snapshot_day_of_week"].isin([5, 6])
        enriched_base_df["is_working_day"] = ~enriched_base_df["is_weekend"]
        enriched_base_df["is_non_working_day"] = enriched_base_df["is_weekend"]
        enriched_base_df["is_holiday"] = False
        enriched_base_df["holiday_type"] = pd.NA
        enriched_base_df["holiday_name"] = pd.NA

    if not weather_for_join_df.empty:
        weather_for_join_df["date"] = parse_datetime_safely(weather_for_join_df["date"]).dt.normalize()

        enriched_base_df = enriched_base_df.merge(
            weather_for_join_df,
            on="date",
            how="left",
        )
    else:
        print("Meteorología limpia no disponible. La base se generará sin variables meteorológicas.")

    enriched_base_output_path.parent.mkdir(parents=True, exist_ok=True)
    processed_modeling_base_output_path.parent.mkdir(parents=True, exist_ok=True)

    enriched_base_df.to_csv(enriched_base_output_path, index=False)
    enriched_base_df.to_csv(processed_modeling_base_output_path, index=False)

    print(f"Archivo intermedio generado: {enriched_base_output_path.relative_to(config.project_root)}")
    print(f"Copia para modelado generada: {processed_modeling_base_output_path.relative_to(config.project_root)}")
    print(f"Filas: {len(enriched_base_df):,}")
    print(f"Columnas: {len(enriched_base_df.columns)}")
    print(f"snapshot_datetime nulos: {int(enriched_base_df['snapshot_datetime'].isna().sum()):,}")

    if "weather_temperature_mean_c" in enriched_base_df.columns:
        print(f"weather_temperature_mean_c nulos: {int(enriched_base_df['weather_temperature_mean_c'].isna().sum()):,}")

    display(enriched_base_df.head())


## 13. Validación de la base enriquecida

Esta sección comprueba que la base enriquecida tiene coherencia mínima antes de pasar al contrato de datos y al EDA.


In [ ]:
if enriched_base_output_path.exists():
    enriched_validation_df = pd.read_csv(
        enriched_base_output_path,
        parse_dates=["snapshot_datetime", "date"],
        low_memory=False,
    )
else:
    enriched_validation_df = pd.DataFrame()

if enriched_validation_df.empty:
    print("No existe base enriquecida para validar.")
else:
    validation_summary = pd.DataFrame(
        [
            {"Métrica": "Filas", "Valor": len(enriched_validation_df)},
            {"Métrica": "Columnas", "Valor": len(enriched_validation_df.columns)},
            {"Métrica": "Fecha mínima snapshot", "Valor": enriched_validation_df["snapshot_datetime"].min()},
            {"Métrica": "Fecha máxima snapshot", "Valor": enriched_validation_df["snapshot_datetime"].max()},
            {"Métrica": "Estaciones históricas únicas", "Valor": enriched_validation_df["station_id_historical"].nunique()},
            {"Métrica": "Nulos en snapshot_datetime", "Valor": int(enriched_validation_df["snapshot_datetime"].isna().sum())},
            {"Métrica": "Nulos en station_id_historical", "Valor": int(enriched_validation_df["station_id_historical"].isna().sum())},
            {"Métrica": "Nulos en num_bikes_available", "Valor": int(enriched_validation_df["num_bikes_available"].isna().sum())},
            {"Métrica": "Nulos en num_docks_available", "Valor": int(enriched_validation_df["num_docks_available"].isna().sum())},
        ]
    )
    display(validation_summary)

    if "weather_temperature_mean_c" in enriched_validation_df.columns:
        print("Variables meteorológicas disponibles en la base enriquecida.")
    else:
        print("Variables meteorológicas no disponibles en la base enriquecida.")


## 14. Contrato de datos para modelado

Esta sección documenta qué archivo debe usar el siguiente notebook y qué columnas principales contiene.

El archivo recomendado para preprocesamiento y modelado es:

`data/processed/station_status_history_2022_modeling_base.csv`

La versión equivalente dentro del flujo intermedio se conserva en:

`data/interim/bicimad/station_status_history_2022_enriched_base.csv`

Esta base no es todavía el dataset final de Machine Learning. Todavía falta crear el target definitivo, decidir variables predictoras, dividir los datos temporalmente y aplicar el preprocesamiento necesario.


In [ ]:
if enriched_validation_df.empty:
    print("No se puede generar contrato de datos porque la base enriquecida no existe todavía.")
else:
    contract_rows = []

    for column in enriched_validation_df.columns:
        contract_rows.append(
            {
                "Columna": column,
                "Tipo pandas": str(enriched_validation_df[column].dtype),
                "Nulos": int(enriched_validation_df[column].isna().sum()),
                "Porcentaje nulos": round(enriched_validation_df[column].isna().mean() * 100, 2),
                "Ejemplo": enriched_validation_df[column].dropna().iloc[0] if enriched_validation_df[column].dropna().shape[0] else pd.NA,
            }
        )

    data_contract_df = pd.DataFrame(contract_rows)
    display(data_contract_df)

    data_contract_path = config.interim_bicimad_dir / "station_status_history_2022_enriched_base_contract.csv"
    data_contract_df.to_csv(data_contract_path, index=False)
    print(f"Contrato de datos generado: {data_contract_path.relative_to(config.project_root)}")


## 15. Reglas de limpieza para EDA

El EDA necesita distinguir entre registros normales y registros donde la estación no está disponible o parece operativamente restringida.

Reglas usadas:

- `available_record`: la estación no está marcada como no disponible.
- `capacity_balance`: capacidad menos bicicletas disponibles menos anclajes disponibles.
- `operationally_constrained`: estación disponible, pero con un balance anómalo alto.
- `normal_operational_record`: registro disponible y no restringido.
- `near_empty`: estación casi vacía.
- `near_full`: estación casi llena.


In [ ]:
if enriched_validation_df.empty:
    eda_df = pd.DataFrame()
    print("No hay base enriquecida para preparar EDA.")
else:
    eda_df = enriched_validation_df.copy()

    for col in ["station_capacity", "num_bikes_available", "num_docks_available", "is_not_available"]:
        if col in eda_df.columns:
            eda_df[col] = pd.to_numeric(eda_df[col], errors="coerce")

    eda_df["available_record"] = eda_df["is_not_available"].fillna(0).eq(0)
    eda_df["capacity_balance"] = (
        eda_df["station_capacity"]
        - eda_df["num_bikes_available"]
        - eda_df["num_docks_available"]
    )

    eda_df["operationally_constrained"] = (
        eda_df["available_record"]
        & (eda_df["station_capacity"].notna())
        & (eda_df["capacity_balance"] >= 0.5 * eda_df["station_capacity"])
    )

    eda_df["normal_operational_record"] = (
        eda_df["available_record"] & ~eda_df["operationally_constrained"]
    )

    eda_df["empty"] = eda_df["normal_operational_record"] & eda_df["num_bikes_available"].eq(0)
    eda_df["near_empty"] = eda_df["normal_operational_record"] & eda_df["num_bikes_available"].le(2)
    eda_df["full"] = eda_df["normal_operational_record"] & eda_df["num_docks_available"].eq(0)
    eda_df["near_full"] = eda_df["normal_operational_record"] & eda_df["num_docks_available"].le(2)

    print("Reglas de EDA aplicadas correctamente.")
    print(f"Filas para EDA: {len(eda_df):,}")


## 16. EDA global

Visión general del estado operativo de la red.


In [ ]:
if eda_df.empty:
    print("No hay datos para EDA global.")
else:
    total_records = len(eda_df)
    available_records = int(eda_df["available_record"].sum())
    normal_records = int(eda_df["normal_operational_record"].sum())

    global_summary = pd.DataFrame(
        [
            {"Métrica": "Registros totales", "Valor": total_records, "Porcentaje": 100.0},
            {"Métrica": "Registros disponibles", "Valor": available_records, "Porcentaje": round(available_records / total_records * 100, 2)},
            {"Métrica": "Registros no disponibles", "Valor": total_records - available_records, "Porcentaje": round((total_records - available_records) / total_records * 100, 2)},
            {"Métrica": "Registros normales para EDA", "Valor": normal_records, "Porcentaje": round(normal_records / total_records * 100, 2)},
            {"Métrica": "Estaciones casi vacías", "Valor": int(eda_df["near_empty"].sum()), "Porcentaje": round(eda_df["near_empty"].mean() * 100, 2)},
            {"Métrica": "Estaciones casi llenas", "Valor": int(eda_df["near_full"].sum()), "Porcentaje": round(eda_df["near_full"].mean() * 100, 2)},
        ]
    )

    display(global_summary)


## 17. EDA por estación

Análisis de estaciones con mayor riesgo de quedarse casi vacías o casi llenas.


In [ ]:
if eda_df.empty:
    print("No hay datos para EDA por estación.")
else:
    station_eda_df = (
        eda_df
        .groupby(["station_id_historical", "station_name"], dropna=False)
        .agg(
            records=("station_id_historical", "count"),
            normal_operational_records=("normal_operational_record", "sum"),
            not_available_pct=("available_record", lambda s: round((1 - s.mean()) * 100, 2)),
            near_empty_pct=("near_empty", lambda s: round(s.mean() * 100, 2)),
            empty_pct=("empty", lambda s: round(s.mean() * 100, 2)),
            near_full_pct=("near_full", lambda s: round(s.mean() * 100, 2)),
            full_pct=("full", lambda s: round(s.mean() * 100, 2)),
            mean_bikes_available=("num_bikes_available", "mean"),
            mean_docks_available=("num_docks_available", "mean"),
            median_capacity=("station_capacity", "median"),
        )
        .reset_index()
    )

    station_eda_df["is_reliable_for_operational_risk"] = (
        (station_eda_df["normal_operational_records"] >= 1000)
        & (station_eda_df["not_available_pct"] <= 20)
    )

    reliable_station_eda_df = station_eda_df[station_eda_df["is_reliable_for_operational_risk"]].copy()

    print("Top estaciones con mayor riesgo de estar casi vacías:")
    display(
        reliable_station_eda_df
        .sort_values("near_empty_pct", ascending=False)
        .head(15)
    )

    print("Top estaciones con mayor riesgo de estar casi llenas:")
    display(
        reliable_station_eda_df
        .sort_values("near_full_pct", ascending=False)
        .head(15)
    )


## 18. EDA temporal

Análisis por hora, mes y tipo de día.


In [ ]:
if eda_df.empty:
    print("No hay datos para EDA temporal.")
else:
    by_hour_df = (
        eda_df
        .groupby("snapshot_hour", dropna=False)
        .agg(
            records=("snapshot_hour", "count"),
            near_empty_pct=("near_empty", lambda s: round(s.mean() * 100, 2)),
            near_full_pct=("near_full", lambda s: round(s.mean() * 100, 2)),
            mean_bikes_available=("num_bikes_available", "mean"),
            mean_docks_available=("num_docks_available", "mean"),
        )
        .reset_index()
    )

    print("Riesgo por hora:")
    display(by_hour_df)

    by_month_df = (
        eda_df
        .groupby("snapshot_month", dropna=False)
        .agg(
            records=("snapshot_month", "count"),
            near_empty_pct=("near_empty", lambda s: round(s.mean() * 100, 2)),
            near_full_pct=("near_full", lambda s: round(s.mean() * 100, 2)),
        )
        .reset_index()
    )

    print("Riesgo por mes:")
    display(by_month_df)

    if "day_type" in eda_df.columns:
        by_day_type_df = (
            eda_df
            .groupby("day_type", dropna=False)
            .agg(
                records=("day_type", "count"),
                near_empty_pct=("near_empty", lambda s: round(s.mean() * 100, 2)),
                near_full_pct=("near_full", lambda s: round(s.mean() * 100, 2)),
            )
            .reset_index()
        )

        print("Riesgo por tipo de día:")
        display(by_day_type_df)


## 19. EDA meteorológica

Análisis sencillo de disponibilidad según precipitación y temperatura.


In [ ]:
if eda_df.empty:
    print("No hay datos para EDA meteorológica.")
elif "weather_temperature_mean_c" not in eda_df.columns:
    print("La base no contiene variables meteorológicas.")
else:
    weather_eda_df = eda_df.copy()

    if "weather_has_precipitation" in weather_eda_df.columns:
        by_precip_df = (
            weather_eda_df
            .groupby("weather_has_precipitation", dropna=False)
            .agg(
                records=("weather_has_precipitation", "count"),
                near_empty_pct=("near_empty", lambda s: round(s.mean() * 100, 2)),
                near_full_pct=("near_full", lambda s: round(s.mean() * 100, 2)),
            )
            .reset_index()
        )

        print("Riesgo según precipitación diaria:")
        display(by_precip_df)

    weather_eda_df["temperature_bucket"] = pd.cut(
        weather_eda_df["weather_temperature_mean_c"],
        bins=[-20, 5, 10, 15, 20, 25, 30, 50],
        labels=["<=5", "5-10", "10-15", "15-20", "20-25", "25-30", ">30"],
    )

    by_temp_df = (
        weather_eda_df
        .groupby("temperature_bucket", observed=False)
        .agg(
            records=("temperature_bucket", "count"),
            near_empty_pct=("near_empty", lambda s: round(s.mean() * 100, 2)),
            near_full_pct=("near_full", lambda s: round(s.mean() * 100, 2)),
        )
        .reset_index()
    )

    print("Riesgo según temperatura media diaria:")
    display(by_temp_df)


## 20. EDA estación + hora

Análisis combinado para detectar estaciones y franjas horarias especialmente críticas.


In [ ]:
if eda_df.empty:
    print("No hay datos para EDA estación + hora.")
else:
    station_hour_df = (
        eda_df
        .groupby(["station_id_historical", "station_name", "snapshot_hour"], dropna=False)
        .agg(
            records=("station_id_historical", "count"),
            normal_operational_records=("normal_operational_record", "sum"),
            near_empty_pct=("near_empty", lambda s: round(s.mean() * 100, 2)),
            empty_pct=("empty", lambda s: round(s.mean() * 100, 2)),
            near_full_pct=("near_full", lambda s: round(s.mean() * 100, 2)),
            full_pct=("full", lambda s: round(s.mean() * 100, 2)),
        )
        .reset_index()
    )

    station_hour_reliable_df = station_hour_df[station_hour_df["normal_operational_records"] >= 100].copy()

    print("Top estación + hora con mayor riesgo de estar casi vacía:")
    display(
        station_hour_reliable_df
        .sort_values("near_empty_pct", ascending=False)
        .head(20)
    )

    print("Top estación + hora con mayor riesgo de estar casi llena:")
    display(
        station_hour_reliable_df
        .sort_values("near_full_pct", ascending=False)
        .head(20)
    )


## 21. Hallazgos principales

Esta sección resume los hallazgos principales detectados por el EDA.

El contenido se genera a partir de las tablas anteriores. Si alguna fuente no se ha podido descargar o limpiar, los hallazgos deberán interpretarse con cautela.


In [ ]:
findings = []

if not eda_df.empty:
    near_empty_pct = round(eda_df["near_empty"].mean() * 100, 2)
    near_full_pct = round(eda_df["near_full"].mean() * 100, 2)

    findings.append(f"El porcentaje global de registros casi vacíos es aproximadamente {near_empty_pct}%.")
    findings.append(f"El porcentaje global de registros casi llenos es aproximadamente {near_full_pct}%.")

    if "by_hour_df" in globals() and not by_hour_df.empty:
        by_hour_valid_df = by_hour_df.dropna(subset=["snapshot_hour"]).copy()

        if not by_hour_valid_df.empty:
            top_empty_hour = by_hour_valid_df.sort_values("near_empty_pct", ascending=False).iloc[0]

            findings.append(
                f"La hora con mayor riesgo global de estación casi vacía es {int(top_empty_hour['snapshot_hour'])}:00 "
                f"con {top_empty_hour['near_empty_pct']}%."
            )
        else:
            findings.append(
                "No se ha podido identificar una hora crítica porque snapshot_hour no contiene valores válidos."
            )

    if "reliable_station_eda_df" in globals() and not reliable_station_eda_df.empty:
        top_empty_station = reliable_station_eda_df.sort_values("near_empty_pct", ascending=False).iloc[0]
        top_full_station = reliable_station_eda_df.sort_values("near_full_pct", ascending=False).iloc[0]

        findings.append(
            f"La estación con mayor riesgo de estar casi vacía es {top_empty_station['station_name']} "
            f"con {top_empty_station['near_empty_pct']}%."
        )

        findings.append(
            f"La estación con mayor riesgo de estar casi llena es {top_full_station['station_name']} "
            f"con {top_full_station['near_full_pct']}%."
        )

if not findings:
    findings.append("No se han podido generar hallazgos porque no hay datos suficientes.")

for index, finding in enumerate(findings, start=1):
    print(f"{index}. {finding}")


## 22. Limitaciones

Principales limitaciones de esta fase:

- El histórico de estaciones corresponde a 2022.
- La red actual de BiciMAD puede no coincidir exactamente con la red histórica.
- Las fuentes GBFS actuales son útiles para referencia o demo, pero no sustituyen al histórico.
- La meteorología se integra a nivel diario, no horario.
- Este notebook no define el target final ni entrena modelos.
- La fase de modelado deberá evitar fuga de información y respetar el orden temporal.


## 23. Entrega para el siguiente notebook

El archivo recomendado para continuar el proyecto es:

`data/processed/station_status_history_2022_modeling_base.csv`

Este archivo debe ser la entrada del siguiente notebook:

`notebooks/02_preprocesamiento.ipynb`

La versión trazable equivalente queda en:

`data/interim/bicimad/station_status_history_2022_enriched_base.csv`

El siguiente notebook debería encargarse de:

1. Definir el target.
2. Decidir si el problema será de regresión o clasificación.
3. Definir horizonte temporal.
4. Crear variables predictoras.
5. Separar entrenamiento y prueba respetando el tiempo.
6. Aplicar encoding, escalado u otros pasos de preprocesamiento.
7. Entrenar modelos.
8. Evaluar resultados.

Esta entrega no debe interpretarse como modelo final ni como dataset final entrenable. Es la base limpia y enriquecida sobre la que modelado debe construir el problema predictivo.


In [ ]:
handoff_summary = pd.DataFrame(
    [
        {
            "Elemento": "Archivo recomendado para modelado",
            "Valor": "data/processed/station_status_history_2022_modeling_base.csv",
        },
        {
            "Elemento": "Archivo intermedio trazable",
            "Valor": "data/interim/bicimad/station_status_history_2022_enriched_base.csv",
        },
        {
            "Elemento": "Tipo de archivo",
            "Valor": "Base histórica limpia y enriquecida",
        },
        {
            "Elemento": "Uso recomendado",
            "Valor": "Entrada para preprocesamiento y modelado",
        },
        {
            "Elemento": "Granularidad",
            "Valor": "Una fila por estación y momento temporal",
        },
        {
            "Elemento": "Identificador principal",
            "Valor": "station_id_historical",
        },
        {
            "Elemento": "Fecha/hora principal",
            "Valor": "snapshot_datetime",
        },
        {
            "Elemento": "No contiene",
            "Valor": "Target final, train/test split, escalado, encoding ni modelo",
        },
        {
            "Elemento": "Notebook siguiente",
            "Valor": "notebooks/02_preprocesamiento.ipynb",
        },
    ]
)

display(handoff_summary)


## 24. Archivos generados

Esta sección lista los archivos principales generados por el notebook.


In [ ]:
generated_files = [
    config.raw_bicimad_dir / "gbfs_station_information_raw.json",
    config.raw_bicimad_dir / "gbfs_station_status_snapshot_raw.json",
    config.raw_bicimad_dir / "bicimad_daily_trips_raw.csv",
    config.raw_bicimad_dir / "bicimad_station_status_history_2022_raw.zip",
    config.raw_calendar_dir / "madrid_working_calendar_raw.csv",
    config.raw_weather_dir / "aemet_stations_inventory_raw.json",
    config.raw_weather_dir / "aemet_daily_climate_selected_stations_2022_raw.json",
    config.raw_weather_dir / "aemet_forecast_hourly_madrid_raw.json",
    config.interim_bicimad_dir / "stations_clean.csv",
    config.interim_bicimad_dir / "station_status_snapshot_clean.csv",
    config.interim_bicimad_dir / "bicimad_daily_trips_clean.csv",
    config.interim_calendar_dir / "madrid_working_calendar_clean.csv",
    config.interim_weather_dir / "aemet_stations_inventory_clean.csv",
    config.interim_weather_dir / "aemet_daily_climate_selected_stations_2022_clean.csv",
    config.interim_weather_dir / "aemet_daily_weather_madrid_2022_join_ready.csv",
    config.interim_weather_dir / "aemet_forecast_hourly_madrid_clean.csv",
    config.interim_bicimad_dir / "station_status_history_2022_clean.csv",
    config.interim_bicimad_dir / "station_status_history_2022_enriched_base.csv",
    config.processed_dir / "station_status_history_2022_modeling_base.csv",
    config.interim_bicimad_dir / "station_status_history_2022_enriched_base_contract.csv",
]

generated_files_df = pd.DataFrame(
    [
        {
            "Archivo": str(path.relative_to(config.project_root)),
            "Existe": path.exists(),
            "Tamaño MB": round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else 0,
        }
        for path in generated_files
    ]
)

display(generated_files_df)

print("Notebook finalizado.")
print("Recuerda: los datos están protegidos por .gitignore y no se suben al repositorio.")
